# Pilot 01 — Evidence Conflict in VLM Damage Assessment

**Question:** when a satellite image and a textual report about the same building disagree, which one does the VLM trust?

Data: subset of my SAR–Optical Earthquake Damage Dataset (İslahiye, field-validated).
Model: Qwen3-VL-2B-Instruct, inference only, deterministic decoding.

Runs top-to-bottom on a free Colab GPU. Full design notes: `experiments/pilot01-evidence-conflict/README.md` in the GeoReason repo.

In [ ]:
!pip -q install "transformers>=4.57.0" accelerate pillow

## 1. Get the code and data

Upload `satdemo.zip` when prompted. The experiment scripts are pulled from the GeoReason repo.

In [ ]:
!git clone -q https://github.com/FatmaElik/GeoReason.git
%cd GeoReason/experiments/pilot01-evidence-conflict

from google.colab import files
import zipfile

uploaded = files.upload()  # upload satdemo.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall(".")
print("Data ready.")

## 2. Prepare samples

Derives the patch class from each mask and draws a red **box outline** around the damage (visual prompt — pixels stay visible).

In [ ]:
!python prepare_samples.py --base satdemo --out pilot_data

In [ ]:
# sanity check: look at one prompt image
import pandas as pd
from PIL import Image
from IPython.display import display

samples = pd.read_csv("pilot_data/samples.csv", dtype={"sample_id": str})
print(samples["true_class"].value_counts())
display(Image.open("pilot_data/" + samples.iloc[5]["prompt_image"]))

## 3. Run the experiment

~200 inference calls (41 images × 5 conditions). Results are saved after **every** call, so a disconnect loses nothing — just re-run the cell and it resumes.

In [ ]:
!python run_experiment.py --data pilot_data

## 4. Analyze

Accuracy per condition, **flip rate** (the core number), abstention usage, and the summary figure.

In [ ]:
!python analyze_results.py --data pilot_data

In [ ]:
from IPython.display import Image as IPImage
IPImage("pilot_data/accuracy_per_condition.png")

## 5. Read the failures, not just the numbers

The most useful part of a pilot: look at individual conflict cases where the model flipped.

In [ ]:
import pandas as pd
from prompts import CLASS_NAMES

df = pd.read_csv("pilot_data/results.csv", dtype={"sample_id": str})
df = df.drop_duplicates(subset=["sample_id", "condition"], keep="last")

base = df[df.condition == "A_image_only"][["sample_id", "answer"]].rename(columns={"answer": "image_only_answer"})
conflict = df[df.condition == "C_text_conflict"].merge(base, on="sample_id")
conflict["true_name"] = conflict["true_class"].map(CLASS_NAMES)

flipped = conflict[
    (conflict.image_only_answer == conflict.true_name) &
    (conflict.answer != conflict.image_only_answer)
]
print(f"Model was right from the image alone, then followed the wrong report: {len(flipped)} cases")
flipped[["sample_id", "true_name", "image_only_answer", "answer", "raw_answer"]].head(10)